#1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re, os, time, threading, multiprocessing
import concurrent.futures
import psutil, warnings
import statistics as stats

In [ ]:
# ---- Mount Google Drive (Colab) ----
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.1 Merge CSV Files

Find the file path for "raw_data.csv"

In [ ]:
!find /content/drive/MyDrive -name "raw_data.csv"    #find file path

/content/drive/MyDrive/Y3S2/HPDP_PROJECT/HPDP_PROJECT_DATA/raw_data.csv


In [ ]:
# ---- Filenames ----
CSV_FILES = [
    'malay_mail_sports.csv',
    'malay_mail_showbiz.csv',
    'world-life-eatdrink.csv',
    'malaysia_money_singapore.csv',
    'tech-gadgets.csv',
]

# ---- Set this to Google Drive folder path ----
DRIVE_FOLDER = '/content/drive/MyDrive/Y3S2/HPDP_PROJECT/HPDP_PROJECT_DATA/' #paste file path at here
RAW_PATH = DRIVE_FOLDER + 'raw_data.csv'

import os, pandas as pd

if not os.path.exists(RAW_PATH):
    frames = [pd.read_csv(DRIVE_FOLDER + f) for f in CSV_FILES
              if os.path.exists(DRIVE_FOLDER + f)]
    if frames:
        merged = pd.concat(frames, ignore_index=True)
        merged.to_csv(RAW_PATH, index=False)
        print(f'Merged {len(frames)} files -> {RAW_PATH}  ({len(merged):,} rows)')
    else:
        print('No source CSVs found. Check DRIVE_FOLDER path and filenames.')
else:
    print(f'Raw file already exists: {RAW_PATH}')

df_peek = pd.read_csv(RAW_PATH, nrows=2)
n_total = sum(1 for _ in open(RAW_PATH)) - 1
print(f'Columns : {list(df_peek.columns)}')
print(f'Rows    : {n_total:,}')


Raw file already exists: /content/drive/MyDrive/Y3S2/HPDP_PROJECT/HPDP_PROJECT_DATA/raw_data.csv
Columns : ['id', 'title', 'category', 'day', 'date', 'time', 'author', 'content', 'link']
Rows    : 154,823


## 1.2 Performance Monitor

PerfMonitor is a lightweight performance monitoring utility
used to track CPU usage and RAM consumption during code execution.

Collects:
- avg_cpu : Average CPU utilization (%)
-  peak_mem_mb : Maximum RAM usage (MB)

In [ ]:
class PerfMonitor:
    def __init__(self, interval=0.1):
        self.interval = interval
        self._cpu, self._mem = [], []
        self._running = False
        self._thread = None

    def _sample(self):
        proc = psutil.Process(os.getpid())
        while self._running:
            self._cpu.append(psutil.cpu_percent(interval=None))
            self._mem.append(proc.memory_info().rss / 1e6)
            time.sleep(self.interval)

    def start(self):
        self._cpu.clear(); self._mem.clear()
        self._running = True
        self._thread = threading.Thread(target=self._sample, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread: self._thread.join()

    @property
    def avg_cpu(self): return float(np.mean(self._cpu)) if self._cpu else 0.0  # Average CPU

    @property
    def peak_mem_mb(self): return float(np.max(self._mem)) if self._mem else 0.0 # Peak memory

    def report(self, label=''):
        print(f'  [{label}] avg CPU : {self.avg_cpu:.1f} %')
        print(f'  [{label}] peak RAM: {self.peak_mem_mb:.1f} MB')


RESULTS = {}  # populated by each pipeline section
print('PerfMonitor ready.')

PerfMonitor ready.


# 2. Baseline Pipeline - Pure Pandas

Cleaning process include:
1. Fix encoding artefacts
2. Remove duplicates
3. Standardise fields
4. Extract location
5. Handle missing values
6. Validate data types



In [ ]:
# ---- Shared cleaning helpers ----

ENCODING_FIXES = [
    ('\u00e2\u20ac\u02dc', "'"),  # fix broken apostrophe encoding
    ('\u00e2\u20ac\u2122', "'"),  # fix right single quote encoding
    ('\u00e2\u20ac\u0153', '"'),  # fix opening quote
    ('\u00e2\u20ac', '"'),        # generic corrupted UTF-8 artifact
    ('\u00e2\u20ac"', ' - '),     # fix malformed dash-like encoding
]


def fix_encoding(series):
    """Repair common latin-1->UTF-8 mojibake."""
    # convert to string, force latin1 encode/decode to repair broken text
    s = (series.astype(str)
               .str.encode('latin1', errors='ignore')
               .str.decode('utf-8', errors='ignore'))

    # replace known corrupted patterns with clean characters
    for bad, good in ENCODING_FIXES:
        s = s.str.replace(bad, good, regex=False)
    return s

# Regex to extract: "location, date — content"
_CONTENT_RE = re.compile(
    r'^([^,]+),\s*([A-Za-z]+\s+\d{1,2})(?:\s*[\u2014\u2013-]\s*|\s+)(.*)$'
)


def split_content(text):
    """Return (location, clean_content) from a raw content string."""
    # handle missing or null values
    if pd.isna(text) or str(text).strip() in ('', 'nan'):
        return None, text

    # normalize spaces
    text = str(text).replace('\xa0', ' ').strip()

    # case 1: match structured pattern using regex
    m = _CONTENT_RE.match(text)
    if m:
        return m.group(1).strip(), m.group(3).strip()

    # case 2: fallback split by first comma
    if ',' in text:
        loc, rest = text.split(',', 1)
        return loc.strip(), rest.strip()

    # case 3: no structure found
    return None, text


def clean_dataframe(df):
    """
    Full cleaning pipeline applied to a pandas DataFrame.
    Steps: A encoding  B dedup  C standardise  D location  E nulls  F types
    """
    df = df.copy()  # avoid modifying original dataframe

    # A: Fix encoding
    for col in ['title', 'content', 'author']:
        if col in df.columns:
            df[col] = fix_encoding(df[col])

    # B: Remove duplicates
    before = len(df)
    df.drop_duplicates(inplace=True)

    # also remove duplicates based on title if available
    if 'title' in df.columns:
        df.drop_duplicates(subset=['title'], keep='first', inplace=True)
    print(f'    Duplicates removed : {before - len(df):,}')

    # C: Standardise fields
    # clean author field (remove "By ", fix null-like values)
    if 'author' in df.columns:
        df['author'] = (df['author'].astype(str)
                        .str.replace(r'^By\s+', '', case=False, regex=True)
                        .str.strip()
                        .replace({'N/A': None, 'nan': None, '': None}))
    # normalize category to lowercase
    if 'category' in df.columns:
        df['category'] = df['category'].astype(str).str.lower().str.strip()
    # trim whitespace in title
    if 'title' in df.columns:
        df['title'] = df['title'].astype(str).str.strip()

    # D: Extract location from content
    if 'content' in df.columns:

        # apply parser row-wise -> returns (location, cleaned content)
        parsed = df['content'].apply(lambda x: pd.Series(split_content(x)))

        df['location'] = parsed.iloc[:, 0]
        df['content']  = parsed.iloc[:, 1]

        # convert location to lowercase
        df['location'] = df['location'].astype('string').str.lower()

        # reposition location column next to author
        cols = list(df.columns)
        if 'location' in cols: cols.remove('location')
        if 'author' in cols:
            cols.insert(cols.index('author') + 1, 'location')
        df = df[cols]

    # E: Handle missing values
    # drop rows where critical fields are missing
    critical = [c for c in ['title', 'content'] if c in df.columns]
    before2 = len(df)
    df.dropna(subset=critical, inplace=True)
    print(f'    Rows dropped (null critical): {before2 - len(df):,}')

    # fill non-critical missing values with "Unknown"
    for col in ['author', 'location', 'category']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')

    # F: Validate data types
    # convert possible date columns into datetime format
    for dcol in ['date', 'date_published', 'published_at', 'publish_date']:
        if dcol in df.columns:
            df[dcol] = pd.to_datetime(df[dcol], errors='coerce')
            print(f'    {dcol}: {df[dcol].isna().sum():,} unparseable -> NaT')

    # enforce string type for text columns
    for scol in ['title', 'content', 'author', 'category', 'location']:
        if scol in df.columns:
            df[scol] = df[scol].astype('string')

    return df


print('Cleaning helpers defined.')

Cleaning helpers defined.


## 2.1 Pandas Baseline Pipeline (5 Runs Benchmark)

In [ ]:
runs_baseline = []  # store results of all 5 runs

print('=' * 70)
print('BASELINE -- Pure Pandas (5 Runs)')
print('=' * 70)

for i in range(5):

    print(f'\nRun {i+1}/5')

    # start performance monitor (CPU + RAM tracking)
    mon_base = PerfMonitor()
    mon_base.start()

    t0 = time.perf_counter()   # start timer

    # -----------------------------
    # Pipeline
    # -----------------------------
    df_raw = pd.read_csv(RAW_PATH, low_memory=False)  # Load raw dataset
    n_raw = len(df_raw)

    df_clean = clean_dataframe(df_raw)

    t1 = time.perf_counter()  # end timer

    mon_base.stop() # stop performance monitor

    # compute runtime and throughput
    elapsed = t1 - t0
    throughput = n_raw / elapsed

    run_result_pipeline = {
        'Run': f'Run {i+1}',            # run label
        'Rows In': n_raw,               # input rows
        'Rows Out': len(df_clean),      # output rows after cleaning
        'Time (s)': round(elapsed, 4),  # total execution time
        'Avg CPU (%)': round(mon_base.avg_cpu, 1),        # avg CPU usage
        'Peak RAM (MB)': round(mon_base.peak_mem_mb, 1),  # peak memory usage
        'Throughput (rps)': round(throughput, 0),         # records processed/sec
    }

    runs_baseline.append(run_result_pipeline)

# ---------------------------------------------------
# Sample Output
# ---------------------------------------------------

sample_df = df_clean.head(5)

print('\n' + '=' * 70)
print('SAMPLE CLEANED OUTPUT')
print('=' * 70)

display(sample_df)

# ---------------------------------------------------
# Save final cleaned dataset to Google Drive
# ---------------------------------------------------
CLEAN_PATH = DRIVE_FOLDER + 'clean_data_pandas.csv'

df_clean.to_csv(CLEAN_PATH, index=False)

print('\n' + '=' * 70)
print(f'Clean data saved to: {CLEAN_PATH}')
print('=' * 70)

BASELINE -- Pure Pandas (5 Runs)

Run 1/5
    Duplicates removed : 1,705
    Rows dropped (null critical): 0


/tmp/ipykernel_4837/3058388662.py:123: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[dcol] = pd.to_datetime(df[dcol], errors='coerce')


    date: 3,512 unparseable -> NaT

Run 2/5
    Duplicates removed : 1,705
    Rows dropped (null critical): 0


/tmp/ipykernel_4837/3058388662.py:123: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[dcol] = pd.to_datetime(df[dcol], errors='coerce')


    date: 3,512 unparseable -> NaT

Run 3/5
    Duplicates removed : 1,705
    Rows dropped (null critical): 0


...

    date: 3,512 unparseable -> NaT

Run 5/5
    Duplicates removed : 1,705
    Rows dropped (null critical): 0


/tmp/ipykernel_4837/3058388662.py:123: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[dcol] = pd.to_datetime(df[dcol], errors='coerce')


    date: 3,512 unparseable -> NaT

SAMPLE CLEANED OUTPUT


,id,title,category,day,date,time,author,location,content,link
0,220214,BAM president: Players flouting rules risk sus...,sports,Saturday,2026-05-16,8:20 PM MYT,Malay Mail,kuala lumpur,National badminton players who violate Badmint...,https://www.malaymail.com/news/sports/2026/05/...
1,220270,Ronaldo left waiting for silverware again afte...,sports,Sunday,2026-05-17,1:10 PM MYT,Unknown,riyadh,Cristiano Ronaldo suffered heartache for the s...,https://www.malaymail.com/news/sports/2026/05/...
2,220227,Man City beat Chelsea 1-0 in FA Cup final than...,sports,Sunday,2026-05-17,9:56 AM MYT,Unknown,london,Antoine Semenyo produced one of the great FA C...,https://www.malaymail.com/news/sports/2026/05/...
3,220226,Small dream: Bernardo Silva refuses to give up...,sports,Sunday,2026-05-17,10:22 AM MYT,Unknown,london,Bernardo Silva said Manchester City still had ...,https://www.malaymail.com/news/sports/2026/05/...
4,220223,"Iran name 30-man World Cup training squad, set...",sports,Sunday,2026-05-17,9:09 AM MYT,Unknown,tehran,Irans World Cup squad will travel to Turkey on...,https://www.malaymail.com/news/sports/2026/05/...



Clean data saved to: /content/drive/MyDrive/Y3S2/HPDP_PROJECT/HPDP_PROJECT_DATA/clean_data_pandas.csv


## 2.2 Result Pandas Baseline Pipeline

In [ ]:
# ---------------------------------------------------
# Baseline Runs Table (5 times)
# ---------------------------------------------------

runs_df = pd.DataFrame(runs_baseline)

print('\n' + '=' * 70)
print('ALL 5 RUNS')
print('=' * 70)

display(runs_df)


ALL 5 RUNS


,Run,Rows In,Rows Out,Time (s),Avg CPU (%),Peak RAM (MB),Throughput (rps)
0,Run 1,123501,121796,25.8236,61.9,1872.0,4782.0
1,Run 2,123501,121796,25.4457,63.1,2276.5,4854.0
2,Run 3,123501,121796,25.7103,54.5,2570.5,4804.0
3,Run 4,123501,121796,26.1038,60.8,2595.5,4731.0
4,Run 5,123501,121796,24.1501,53.2,2609.1,5114.0


## 2.3 Average Result Pandas Baseline Pipeline

In [ ]:
# ---------------------------------------------------
# Remove fastest + slowest run
# ---------------------------------------------------

sorted_runs = sorted(runs_baseline, key=lambda x: x['Time (s)'])

filtered_runs = sorted_runs[1:-1]

# ---------------------------------------------------
# Average Remaining 3 Runs
# ---------------------------------------------------

avg_result_pipeline = {
    'Rows In': filtered_runs[0]['Rows In'],
    'Rows Out': filtered_runs[0]['Rows Out'],
    'Time (s)': round(stats.mean(r['Time (s)'] for r in filtered_runs), 4),
    'Avg CPU (%)': round(stats.mean(r['Avg CPU (%)'] for r in filtered_runs), 1),
    'Peak RAM (MB)': round(stats.mean(r['Peak RAM (MB)'] for r in filtered_runs), 1),
    'Throughput (rps)': round(stats.mean(r['Throughput (rps)'] for r in filtered_runs), 0),
}

summary_df_pipeline = pd.DataFrame([avg_result_pipeline], index=['Average (Min/Max Removed)'])

# ---------------------------------------------------
# Final Averaged Result Table
# ---------------------------------------------------

print('\n' + '=' * 70)
print('FINAL AVERAGED PERFORMANCE')
print('=' * 70)

display(summary_df_pipeline)


FINAL AVERAGED PERFORMANCE


,Rows In,Rows Out,Time (s),Avg CPU (%),Peak RAM (MB),Throughput (rps)
Average (Min/Max Removed),123501,121796,25.6599,59.8,2239.7,4813.0


## 2.4 Save Performance Result in CSV file

In [ ]:
final_df = pd.concat(
    [
        runs_df,
        summary_df_pipeline.reset_index().rename(
            columns={'index': 'Run'}
        )
    ],
    ignore_index=True
)

PERF_PATH = DRIVE_FOLDER + 'pandas_performance_results.csv'

final_df.to_csv(PERF_PATH, index=False)

print(f'Saved: {PERF_PATH}')

display(final_df)

Saved: /content/drive/MyDrive/Y3S2/HPDP_PROJECT/HPDP_PROJECT_DATA/pandas_performance_results.csv


,Run,Rows In,Rows Out,Time (s),Avg CPU (%),Peak RAM (MB),Throughput (rps)
0,Run 1,123501,121796,25.8236,61.9,1872.0,4782.0
1,Run 2,123501,121796,25.4457,63.1,2276.5,4854.0
2,Run 3,123501,121796,25.7103,54.5,2570.5,4804.0
3,Run 4,123501,121796,26.1038,60.8,2595.5,4731.0
4,Run 5,123501,121796,24.1501,53.2,2609.1,5114.0
5,Average (Min/Max Removed),123501,121796,25.6599,59.8,2239.7,4813.0


# 3. Baseline Pipeline - Dask

In [ ]:
import dask.dataframe as dd
import pandas as pd
import time
import re

# ===================================================
# DASK CLEANING HELPER FUNCTIONS
# ===================================================
ENCODING_FIXES = [
    ('\u00e2\u20ac\u02dc', "'"),
    ('\u00e2\u20ac\u2122', "'"),
    ('\u00e2\u20ac\u0153', '"'),
    ('\u00e2\u20ac', '"'),
    ('\u00e2\u20ac"', ' - '),
]

_CONTENT_RE = re.compile(
    r'^([^,]+),\s*([A-Za-z]+\s+\d{1,2})(?:\s*[\u2014\u2013-]\s*|\s+)(.*)$'
)

def split_content_dask(text):
    if text is None or str(text).strip() in ('', 'nan'):
        return {"location": None, "content": text}

    text = str(text).replace('\xa0', ' ').strip()

    m = _CONTENT_RE.match(text)
    if m:
        return {
            "location": m.group(1).strip().lower(),
            "content": m.group(3).strip()
        }

    if ',' in text:
        loc, rest = text.split(',', 1)
        return {
            "location": loc.strip().lower(),
            "content": rest.strip()
        }

    return {"location": None, "content": text}


def clean_dataframe_dask(df: dd.DataFrame) -> dd.DataFrame:

    # ===================================================
    # A: Fix encoding
    # ===================================================
    for col in ['title', 'content', 'author']:
        if col in df.columns:
            for bad, good in ENCODING_FIXES:
                df[col] = df[col].str.replace(
                    bad,
                    good,
                    regex=False
                )

    # ===================================================
    # B: Remove duplicates
    # ===================================================
    before = df.shape[0].compute()

    df = df.drop_duplicates()

    if 'title' in df.columns:
        df = df.drop_duplicates(subset=['title'])

    after = df.shape[0].compute()

    print(f'    Duplicates removed : {before - after:,}')

    # ===================================================
    # C: Standardise fields
    # ===================================================
    if 'author' in df.columns:
        df['author'] = (
            df['author']
            .str.replace(r'(?i)^By\s+', '', regex=True)
            .str.strip()
            .replace({
                'N/A': None,
                'nan': None,
                '': None
            })
        )

    if 'category' in df.columns:
        df['category'] = (
            df['category']
            .str.lower()
            .str.strip()
        )

    if 'title' in df.columns:
        df['title'] = df['title'].str.strip()

    # ===================================================
    # D: Extract location from content
    # ===================================================
    if 'content' in df.columns:

        meta = pd.DataFrame({
            'location': pd.Series(dtype='object'),
            'content': pd.Series(dtype='object')
        })

        extracted = df['content'].apply(
            split_content_dask,
            meta=('content', 'object')
        )

        extracted_df = extracted.apply(
            pd.Series,
            meta=meta
        )

        df['location'] = extracted_df['location']
        df['content'] = extracted_df['content']

        # Reorder columns
        cols = list(df.columns)

        if 'location' in cols:
            cols.remove('location')

        if 'author' in cols:
            cols.insert(cols.index('author') + 1, 'location')
        else:
            cols.append('location')

        df = df[cols]

    # ===================================================
    # E: Handle missing values
    # ===================================================
    critical = [c for c in ['title', 'content'] if c in df.columns]

    before2 = df.shape[0].compute()

    df = df.dropna(subset=critical)

    after2 = df.shape[0].compute()

    print(f'    Rows dropped (null critical): {before2 - after2:,}')

    fill_cols = [
        c for c in ['author', 'location', 'category']
        if c in df.columns
    ]

    for c in fill_cols:
        df[c] = df[c].fillna('Unknown')

    # ===================================================
    # F: Validate data types
    # ===================================================
    for scol in ['title', 'content', 'author', 'category', 'location']:
        if scol in df.columns:
            df[scol] = df[scol].astype(str)

    return df

print('Dask cleaning helpers defined.')

Dask cleaning helpers defined.


## 3.1 Dask Baseline Pipeline

In [ ]:
# ===================================================
# DASK BASELINE PIPELINE (5 RUNS BENCHMARK)
# ===================================================
runs_baseline_dask = []

print('=' * 70)
print('BASELINE -- Dask (5 Runs)')
print('=' * 70)

for i in range(5):
    print(f'\nRun {i+1}/5')

    # start performance monitor
    # (Requires the PerfMonitor class from your notebook)
    mon_base = PerfMonitor()
    mon_base.start()

    t0 = time.perf_counter()

    # ---------------------------------------------------
    # Load data
    # assume_missing=True helps prevent dtype issues
    # blocksize can be tuned for performance
    # ---------------------------------------------------
    df_raw = dd.read_csv(
        RAW_PATH,
        dtype=str,
        assume_missing=True,
        blocksize="64MB"
    )

    # Total rows
    n_raw = df_raw.shape[0].compute()

    # ---------------------------------------------------
    # Clean data using Dask helper
    # ---------------------------------------------------
    df_clean = clean_dataframe_dask(df_raw)

    # Force execution
    df_clean = df_clean.persist()

    t1 = time.perf_counter()
    mon_base.stop()

    elapsed = t1 - t0
    throughput = n_raw / elapsed if elapsed > 0 else 0

    # Final output rows
    n_clean = df_clean.shape[0].compute()

    run_result = {
        'Run': f'Run {i+1}',
        'Rows In': n_raw,
        'Rows Out': n_clean,
        'Time (s)': round(elapsed, 4),
        'Avg CPU (%)': round(mon_base.avg_cpu, 1),
        'Peak RAM (MB)': round(mon_base.peak_mem_mb, 1),
        'Throughput (rps)': round(throughput, 0),
    }

    runs_baseline_dask.append(run_result)

# ---------------------------------------------------
# Sample Output
# ---------------------------------------------------
sample_df = df_clean.head(5)

print('\n' + '=' * 70)
print('SAMPLE CLEANED OUTPUT')
print('=' * 70)

# Already pandas because Dask .head() returns pandas DataFrame
display(sample_df)

# ---------------------------------------------------
# Save final cleaned dataset to Google Drive
# ---------------------------------------------------
CLEAN_PATH_DASK = DRIVE_FOLDER + 'clean_data_dask.csv'

# single_file=True creates one CSV instead of partitions
df_clean.to_csv(
    CLEAN_PATH_DASK,
    single_file=True,
    index=False
)

print('\n' + '=' * 70)
print(f'Clean data saved to: {CLEAN_PATH_DASK}')
print('=' * 70)

BASELINE -- Dask (5 Runs)

Run 1/5
    Duplicates removed : 1,704
    Rows dropped (null critical): 1

Run 2/5
    Duplicates removed : 1,704
    Rows dropped (null critical): 1

Run 3/5
    Duplicates removed : 1,704
    Rows dropped (null critical): 1

Run 4/5
    Duplicates removed : 1,704
    Rows dropped (null critical): 1

Run 5/5
    Duplicates removed : 1,704
    Rows dropped (null critical): 1

SAMPLE CLEANED OUTPUT


,id,title,category,day,date,time,author,location,content,link
2,220227,Man City beat Chelsea 1-0 in FA Cup final than...,sports,Sunday,17-May-26,9:56 AM MYT,Unknown,london,Antoine Semenyo produced one of the great FA C...,https://www.malaymail.com/news/sports/2026/05/...
48,219360,Johor Regent honoured with Sports Leadership I...,sports,Saturday,9-May-26,5:40 PM MYT,Unknown,kuala lumpur,"The Regent of Johor, Tunku Mahkota Ismail Sult...",https://www.malaymail.com/news/sports/2026/05/...
60,219184,Import player quota in Malaysia should be cut ...,sports,Friday,8-May-26,1:46 PM MYT,Unknown,kota bharu,The import player quota in domestic competitio...,https://www.malaymail.com/news/sports/2026/05/...
92,218610,Man United seal Champions League return with d...,sports,Monday,4-May-26,9:02 AM MYT,Unknown,manchester,Manchester United secured Champions League foo...,https://www.malaymail.com/news/sports/2026/05/...
154,217792,Malaysian squash champ Sivasangari fights back...,sports,Monday,27-Apr-26,8:43 AM MYT,Unknown,kuala lumpur,National squash queen S. Sivasangari produced ...,https://www.malaymail.com/news/sports/2026/04/...



Clean data saved to: /content/drive/MyDrive/Y3S2/HPDP_PROJECT/HPDP_PROJECT_DATA/clean_data_dask.csv


## 3.2 Result Dask Baseline Pipeline

In [ ]:
# ===================================================
# RESULT DASK BASELINE PIPELINE
# ===================================================
df_res_dask = pd.DataFrame(runs_baseline_dask)

print('\n' + '=' * 70)
print('RESULT DASK BASELINE PIPELINE')
print('=' * 70)

display(df_res_dask)


RESULT DASK BASELINE PIPELINE


,Run,Rows In,Rows Out,Time (s),Avg CPU (%),Peak RAM (MB),Throughput (rps)
0,Run 1,123501,121796,154.5005,70.1,3876.4,799.0
1,Run 2,123501,121796,150.5261,69.7,4747.0,820.0
2,Run 3,123501,121796,155.1933,69.1,4800.3,796.0
3,Run 4,123501,121796,148.4582,68.9,4652.7,832.0
4,Run 5,123501,121796,152.9022,73.1,4815.4,808.0


## 3.3 Average Result Dask Baseline Pipeline

In [ ]:
# ===================================================
# AVERAGE RESULT DASK BASELINE PIPELINE
# ===================================================
print('\n' + '=' * 70)
print('AVERAGE RESULT DASK BASELINE PIPELINE')
print('=' * 70)

# Calculate the mean of the numeric columns
# across the 5 runs
df_avg_dask = pd.DataFrame(
    df_res_dask.mean(numeric_only=True)
).T

display(df_avg_dask)


AVERAGE RESULT DASK BASELINE PIPELINE


,Rows In,Rows Out,Time (s),Avg CPU (%),Peak RAM (MB),Throughput (rps)
0,123501.0,121796.0,152.31606,70.18,4578.36,811.0


## 3.4 Save Performance Result in CSV file

In [ ]:
# ===================================================
# 5. SAVE DASK PERFORMANCE METRICS TO CSV
# ===================================================
import pandas as pd

# Give the average row a name so it looks clean
# in the final table
df_avg_dask['Run'] = 'Average'

# Combine the 5 runs (df_res_dask)
# with the average row (df_avg_dask)
final_dask_perf = pd.concat(
    [
        df_res_dask,
        df_avg_dask
    ],
    ignore_index=True
)

# Set the save path for the metrics
PERF_PATH_DASK = DRIVE_FOLDER + 'dask_performance_results.csv'

# Save to Google Drive
final_dask_perf.to_csv(PERF_PATH_DASK, index=False)

print('\n' + '=' * 70)
print(f'✅ Saved Dask performance metrics to: {PERF_PATH_DASK}')
print('=' * 70)

# Display the final combined table
display(final_dask_perf)


✅ Saved Dask performance metrics to: /content/drive/MyDrive/Y3S2/HPDP_PROJECT/HPDP_PROJECT_DATA/dask_performance_results.csv


,Run,Rows In,Rows Out,Time (s),Avg CPU (%),Peak RAM (MB),Throughput (rps)
0,Run 1,123501.0,121796.0,154.50050,70.10,3876.40,799.0
1,Run 2,123501.0,121796.0,150.52610,69.70,4747.00,820.0
2,Run 3,123501.0,121796.0,155.19330,69.10,4800.30,796.0
3,Run 4,123501.0,121796.0,148.45820,68.90,4652.70,832.0
4,Run 5,123501.0,121796.0,152.90220,73.10,4815.40,808.0
5,Average,123501.0,121796.0,152.31606,70.18,4578.36,811.0


# 4.1 Baseline Pipeline - Polars

In [ ]:
import polars as pl
import pandas as pd
import time
import re

# ===================================================
# POLARS CLEANING HELPER FUNCTIONS
# ===================================================
ENCODING_FIXES = [
    ('\u00e2\u20ac\u02dc', "'"),
    ('\u00e2\u20ac\u2122', "'"),
    ('\u00e2\u20ac\u0153', '"'),
    ('\u00e2\u20ac', '"'),
    ('\u00e2\u20ac"', ' - '),
]

_CONTENT_RE = re.compile(r'^([^,]+),\s*([A-Za-z]+\s+\d{1,2})(?:\s*[\u2014\u2013-]\s*|\s+)(.*)$')

def split_content_polars(text):
    if text is None or str(text).strip() in ('', 'nan'):
        return {"location": None, "content": text}
    text = str(text).replace('\xa0', ' ').strip()
    m = _CONTENT_RE.match(text)
    if m:
        return {"location": m.group(1).strip().lower(), "content": m.group(3).strip()}
    if ',' in text:
        loc, rest = text.split(',', 1)
        return {"location": loc.strip().lower(), "content": rest.strip()}
    return {"location": None, "content": text}

def clean_dataframe_polars(df: pl.DataFrame) -> pl.DataFrame:
    # A: Fix encoding
    replace_exprs = []
    for col in ['title', 'content', 'author']:
        if col in df.columns:
            expr = pl.col(col)
            for bad, good in ENCODING_FIXES:
                expr = expr.str.replace_all(bad, good, literal=True)
            replace_exprs.append(expr)
    if replace_exprs:
        df = df.with_columns(replace_exprs)

    # B: Remove duplicates
    before = df.height
    df = df.unique()
    if 'title' in df.columns:
        df = df.unique(subset=['title'], keep='first')
    print(f'    Duplicates removed : {before - df.height:,}')

    # C: Standardise fields
    if 'author' in df.columns:
        df = df.with_columns(
            pl.col('author').str.replace(r'(?i)^By\s+', '')
                            .str.strip_chars()
                            .replace({'N/A': None, 'nan': None, '': None})
        )
    if 'category' in df.columns:
        df = df.with_columns(pl.col('category').str.to_lowercase().str.strip_chars())
    if 'title' in df.columns:
        df = df.with_columns(pl.col('title').str.strip_chars())

    # D: Extract location from content
    if 'content' in df.columns:
        extracted = df.get_column('content').map_elements(
            split_content_polars,
            return_dtype=pl.Struct([pl.Field("location", pl.String), pl.Field("content", pl.String)])
        )
        df = df.with_columns([
            extracted.struct.field("location").alias("location"),
            extracted.struct.field("content").alias("content")
        ])

        cols = df.columns
        if 'location' in cols: cols.remove('location')
        if 'author' in cols:
            cols.insert(cols.index('author') + 1, 'location')
        df = df.select(cols)

    # E: Handle missing values
    critical = [c for c in ['title', 'content'] if c in df.columns]
    before2 = df.height
    df = df.drop_nulls(subset=critical)
    print(f'    Rows dropped (null critical): {before2 - df.height:,}')

    fill_cols = [c for c in ['author', 'location', 'category'] if c in df.columns]
    if fill_cols:
        df = df.with_columns([pl.col(c).fill_null('Unknown') for c in fill_cols])

    # F: Validate data types
    for scol in ['title', 'content', 'author', 'category', 'location']:
        if scol in df.columns:
            df = df.with_columns(pl.col(scol).cast(pl.String))

    return df

    print('Cleaning helpers defined.')

## 4.2 Polars Baseline Pipeline (5 Runs Benchmark)

In [ ]:
# ===================================================
# POLARS BASELINE PIPELINE (5 RUNS BENCHMARK)
# ===================================================
runs_baseline_polars = []

print('=' * 70)
print('BASELINE -- Polars (5 Runs)')
print('=' * 70)

for i in range(5):
    print(f'\nRun {i+1}/5')

    # start performance monitor (Requires the PerfMonitor class from your notebook)
    mon_base = PerfMonitor()
    mon_base.start()

    t0 = time.perf_counter()

    # Load data (infer_schema_length=0 forces safe string reading)
    df_raw = pl.read_csv(RAW_PATH, infer_schema_length=0)
    n_raw = df_raw.height

    # Clean data using the Polars helper
    df_clean = clean_dataframe_polars(df_raw)

    t1 = time.perf_counter()
    mon_base.stop()

    elapsed = t1 - t0
    throughput = n_raw / elapsed if elapsed > 0 else 0

    run_result = {
        'Run': f'Run {i+1}',
        'Rows In': n_raw,
        'Rows Out': df_clean.height,
        'Time (s)': round(elapsed, 4),
        'Avg CPU (%)': round(mon_base.avg_cpu, 1),
        'Peak RAM (MB)': round(mon_base.peak_mem_mb, 1),
        'Throughput (rps)': round(throughput, 0),
    }
    runs_baseline_polars.append(run_result)

# ---------------------------------------------------
# Sample Output
# ---------------------------------------------------
sample_df = df_clean.head(5)

print('\n' + '=' * 70)
print('SAMPLE CLEANED OUTPUT')
print('=' * 70)

# Convert to Pandas just for the display to force the clean HTML formatting!
display(sample_df.to_pandas())

# ---------------------------------------------------
# Save final cleaned dataset to Google Drive
# ---------------------------------------------------
# Giving it a specific Polars name so it doesn't overwrite your Pandas one
CLEAN_PATH_POLARS = DRIVE_FOLDER + 'clean_data_polars.csv'

df_clean.write_csv(CLEAN_PATH_POLARS)

print('\n' + '=' * 70)
print(f'Clean data saved to: {CLEAN_PATH_POLARS}')
print('=' * 70)

BASELINE -- Polars (5 Runs)

Run 1/5
    Duplicates removed : 1,704
    Rows dropped (null critical): 0

Run 2/5
    Duplicates removed : 1,704
    Rows dropped (null critical): 0

Run 3/5
    Duplicates removed : 1,704
    Rows dropped (null critical): 0

Run 4/5
    Duplicates removed : 1,704
    Rows dropped (null critical): 0

Run 5/5
    Duplicates removed : 1,704
    Rows dropped (null critical): 0

SAMPLE CLEANED OUTPUT


,id,title,category,day,date,time,author,location,content,link
0,996085,"Love or hate the ‘man bun’, get used to it, Go...",life,Friday,30-Oct-15,8:02 AM MYT,Unknown,los angeles,"Love it or hate it, the “man bun” is one of th...",https://www.malaymail.com/news/life/2015/10/30...
1,188138,Segamat hotel staff investigated after flag is...,malaysia,Tuesday,19-Aug-25,9:49 AM MYT,Unknown,segamat,Police have launched an investigation into a s...,https://www.malaymail.com/news/malaysia/2025/0...
2,1761977,Muhammad Ali's autographed gold glove fails to...,life,Friday,14-Jun-19,7:42 AM MYT,Unknown,turin,A rare gold boxing glove made to commemorate M...,https://www.malaymail.com/news/life/2019/06/14...
3,1957048,"‘We’re not racist’, says Prince William after ...",life,Thursday,11-Mar-21,9:35 PM MYT,Unknown,london,Prince William denied today that Britain’s roy...,https://www.malaymail.com/news/life/2021/03/11...
4,1962326,Neelofa makes jaws drop with wedding dress stu...,showbiz,Tuesday,30-Mar-21,3:06 PM MYT,TAN MEI ZI,petaling jaya,Malaysian entrepreneur Neelofa left fans in aw...,https://www.malaymail.com/news/showbiz/2021/03...



Clean data saved to: /content/drive/MyDrive/Y3S2/HPDP_PROJECT/HPDP_PROJECT_DATA/clean_data_polars.csv


## 4.2 Result Polars Baseline Pipeline

In [ ]:
# ===================================================
# RESULT POLARS BASELINE PIPELINE
# ===================================================
df_res_polars = pd.DataFrame(runs_baseline_polars)

print('\n' + '=' * 70)
print('RESULT POLARS BASELINE PIPELINE')
print('=' * 70)
display(df_res_polars)


RESULT POLARS BASELINE PIPELINE


,Run,Rows In,Rows Out,Time (s),Avg CPU (%),Peak RAM (MB),Throughput (rps)
0,Run 1,123501,121797,10.8944,90.2,5204.0,11336.0
1,Run 2,123501,121797,4.9495,61.2,5798.3,24952.0
2,Run 3,123501,121797,5.9388,96.0,6174.0,20796.0
3,Run 4,123501,121797,4.7464,44.4,6447.8,26020.0
4,Run 5,123501,121797,4.5332,55.0,6490.1,27244.0


## 4.3 Average Result Polars Baseline Pipeline

In [ ]:
# ===================================================
# AVERAGE RESULT POLARS BASELINE PIPELINE
# ===================================================
print('\n' + '=' * 70)
print('AVERAGE RESULT POLARS BASELINE PIPELINE')
print('=' * 70)

# Calculate the mean of the numeric columns across the 5 runs
df_avg_polars = pd.DataFrame(df_res_polars.mean(numeric_only=True)).T
display(df_avg_polars)


AVERAGE RESULT POLARS BASELINE PIPELINE


,Rows In,Rows Out,Time (s),Avg CPU (%),Peak RAM (MB),Throughput (rps)
0,123501.0,121797.0,6.21246,69.36,6022.84,22069.6


## 4.4 Save Performance Result in CSV file

In [ ]:
# ===================================================
# 5. SAVE POLARS PERFORMANCE METRICS TO CSV
# ===================================================
import pandas as pd

# Give the average row a name so it looks clean in the final table
df_avg_polars['Run'] = 'Average'

# Combine the 5 runs (df_res_polars) with the average row (df_avg_polars)
final_polars_perf = pd.concat(
    [
        df_res_polars,
        df_avg_polars
    ],
    ignore_index=True
)

# Set the save path for the metrics
PERF_PATH_POLARS = DRIVE_FOLDER + 'polars_performance_results.csv'

# Save to Google Drive
final_polars_perf.to_csv(PERF_PATH_POLARS, index=False)

print('\n' + '=' * 70)
print(f'✅ Saved Polars performance metrics to: {PERF_PATH_POLARS}')
print('=' * 70)

# Display the final combined table
display(final_polars_perf)


✅ Saved Polars performance metrics to: /content/drive/MyDrive/Y3S2/HPDP_PROJECT/HPDP_PROJECT_DATA/polars_performance_results.csv


,Run,Rows In,Rows Out,Time (s),Avg CPU (%),Peak RAM (MB),Throughput (rps)
0,Run 1,123501.0,121797.0,10.89440,90.20,5204.00,11336.0
1,Run 2,123501.0,121797.0,4.94950,61.20,5798.30,24952.0
2,Run 3,123501.0,121797.0,5.93880,96.00,6174.00,20796.0
3,Run 4,123501.0,121797.0,4.74640,44.40,6447.80,26020.0
4,Run 5,123501.0,121797.0,4.53320,55.00,6490.10,27244.0
5,Average,123501.0,121797.0,6.21246,69.36,6022.84,22069.6
